<a href="https://colab.research.google.com/github/jellyandcream494/Transformer-Component-Selection/blob/main/transformerexperiment_baseline_v2_3models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Imports and Hyperparameters
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import time
import math
import urllib.request
import os

# Set device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# --- Hyperparameters (Scaled up for Colab T4) ---
# T4 has 16GB VRAM, allowing us to push these significantly higher than an M2 Air
batch_size = 64      # Increased batch size for T4 GPU utilization
block_size = 256     # Context window size (doubled)
d_model = 384        # Embedding dimension (tripled)
n_heads = 6          # Number of attention heads
n_layers = 6         # Number of transformer blocks (doubled)
dropout = 0.2        # Increased dropout to prevent overfitting with a larger model
learning_rate = 5e-4 # Slightly lower LR for larger, deeper model
max_iters = 1500     # ~5-8 mins on T4
eval_interval = 250  # How often to check validation loss
eval_iters = 20      # How many batches to use for evaluation

torch.manual_seed(1337) # For reproducibility

Using device: cuda


In [2]:
# Cell 2: Data Ingestion & Setup
url = "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt"
file_path = "wikitext-2-train.txt"

if not os.path.exists(file_path):
    print("Downloading WikiText-2 dataset...")
    urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

# Since we want to train fast, let's use a subset of the data (e.g., first 1 million characters)
text = text[:1000000]

# Extremely simple Character-level Tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Unique characters (vocab size): {vocab_size}")
print(f"Dataset length: {len(text)} characters")

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Create train/val splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # Generates a small batch of data of inputs (x) and targets (y)
    data_source = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print(f"Input batch shape: {xb.shape}")
print(f"Target batch shape: {yb.shape}")

Unique characters (vocab size): 150
Dataset length: 1000000 characters
Input batch shape: torch.Size([64, 256])
Target batch shape: torch.Size([64, 256])


In [3]:
# Cell 3: Core Components (Feed Forward Network)

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(), # GELU is standard for modern transformers like GPT
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [4]:
# Cell 4: Modular Transformer Block

class Block(nn.Module):
    """ Transformer block: communication (attention) followed by computation (FFN) """

    def __init__(self, n_embd, n_heads, attention_layer):
        super().__init__()
        # Here we inject the specific attention mechanism
        self.sa = attention_layer
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # We use standard residual connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [5]:
# Cell 5: Baseline Multi-Head Attention (MHA)

class MultiHeadAttention(nn.Module):
    """ Standard Scaled Dot-Product Multi-Head Attention """

    def __init__(self, n_heads, n_embd):
        super().__init__()
        assert n_embd % n_heads == 0, "Embedding dimension must be divisible by number of heads"
        self.n_heads = n_heads
        self.d_k = n_embd // n_heads # Head dimension

        # Key, Query, Value projections
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Create a causal lower-triangular mask
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                    .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length (Time), Embedding dims (Channels)

        # Calculate Query, Key, Value for all heads in batch
        # Output shape: (B, T, 3 * C)
        qkv = self.c_attn(x)

        # Split Q, K, V -> each is (B, T, C)
        q, k, v = qkv.split(d_model, dim=2)

        # Reshape to (B, n_heads, T, d_k)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Causal logic: Q x K^T / sqrt(d_k)
        wei = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_k))

        # Apply causal mask (so it can't look ahead at future tokens)
        wei = wei.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)

        # Apply attention to values
        out = wei @ v # (B, n_heads, T, T) @ (B, n_heads, T, d_k) -> (B, n_heads, T, d_k)

        # Re-assemble all head outputs side by side
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        # Final projection back to C dimensions
        out = self.resid_dropout(self.c_proj(out))
        return out

In [6]:
# Cell 6: Language Model Wrapper

class ModularGPT(nn.Module):
    def __init__(self, attention_class):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, d_model)
        self.position_embedding_table = nn.Embedding(block_size, d_model)

        # We instantiate a list of Blocks, optionally passing our specific attention class
        self.blocks = nn.Sequential(*[
            Block(d_model, n_heads, attention_class(n_heads, d_model))
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model) # Final layer norm
        self.lm_head = nn.Linear(d_model, vocab_size) # Language model head (predictions)

        # Initialize weights (standard GPT initialization)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)

        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens to prevent out-of-bounds positioning
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [7]:
# Cell 7: Training Engine

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def train_model(model_name, attention_class):
    print(f"\n--- Training Model with {model_name} ---")
    print(f"Config: block_size={block_size}, batch_size={batch_size}, d_model={d_model}, layers={n_layers}")

    # We pass the class in dynamically so the model builds it internally!
    model = ModularGPT(attention_class).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    start_time = time.time()

    for iter in range(max_iters):
        # Evaluate context occasionally
        if iter % eval_interval == 0 or iter == max_iters - 1:
            losses = estimate_loss(model)
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # Sample batch
        xb, yb = get_batch('train')

        # Evaluate loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    end_time = time.time()
    execution_time = end_time - start_time

    # Final eval
    final_losses = estimate_loss(model)
    val_loss = final_losses['val']
    perplexity = math.exp(val_loss)

    print(f"Training completed in {execution_time:.2f} seconds.")
    print(f"Final Val Loss: {val_loss:.4f} | Perplexity: {perplexity:.4f}")

    # Generate sample
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    sample_text = decode(model.generate(context, max_new_tokens=100)[0].tolist())
    print("\nSample Output:\n", sample_text.replace('\n', ' '))

    return {
        "Model": model_name,
        "Block Size": block_size,
        "Batch Size": batch_size,
        "Val Loss": val_loss,
        "Perplexity": perplexity,
        "Time (s)": execution_time
    }

# Let's test the baseline!
results_registry = []
mha_results = train_model("Multi-Head Attention (Baseline)", MultiHeadAttention)
results_registry.append(mha_results)


--- Training Model with Multi-Head Attention (Baseline) ---
Config: block_size=256, batch_size=64, d_model=384, layers=6
step 0: train loss 5.1891, val loss 5.1783
step 250: train loss 2.4287, val loss 2.4438
step 500: train loss 2.2299, val loss 2.2724
step 750: train loss 1.8301, val loss 1.9004
step 1000: train loss 1.5265, val loss 1.6469
step 1250: train loss 1.3319, val loss 1.5162
step 1499: train loss 1.2201, val loss 1.4588
Training completed in 815.03 seconds.
Final Val Loss: 1.4637 | Perplexity: 4.3218

Sample Output:
       Egyptian to tapprange , which are public arsenal found such at Late <unk> in his bakeross .   T


In [8]:
# Cell 8: Multi-Query Attention (MQA)

class MultiQueryAttention(nn.Module):
    """ Multi-Query Attention (MQA) - Multiple query heads, single shared KV head """

    def __init__(self, n_heads, n_embd):
        super().__init__()
        assert n_embd % n_heads == 0, "Embedding dimension must be divisible by number of heads"
        self.n_heads = n_heads
        self.d_k = n_embd // n_heads # Head dimension

        # Query projection (multiple heads)
        self.c_q = nn.Linear(n_embd, n_embd, bias=False)
        # Key/Value projection (single head shared across all queries)
        self.c_kv = nn.Linear(n_embd, 2 * self.d_k, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Create a causal lower-triangular mask
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                    .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length, Embedding dim

        # Queries for all heads: (B, T, C) -> reshape to (B, T, n_heads, d_k)
        q = self.c_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Key & Value for single head: (B, T, 2 * d_k)
        kv = self.c_kv(x)
        k, v = kv.split(self.d_k, dim=2)

        # Reshape to (B, 1, T, d_k) so they broadcast automatically against Q's (B, n_heads, T, d_k)
        k = k.view(B, T, 1, self.d_k).transpose(1, 2)
        v = v.view(B, T, 1, self.d_k).transpose(1, 2)

        # Causal logic: Q (B, n_heads, T, d_k) @ K^T (B, 1, d_k, T) -> (B, n_heads, T, T)
        wei = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_k))

        # Apply causal mask
        wei = wei.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)

        # Apply attention to values: wei (B, n_heads, T, T) @ v (B, 1, T, d_k) -> out (B, n_heads, T, d_k)
        out = wei @ v

        # Re-assemble all head outputs side by side
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        # Final projection
        out = self.resid_dropout(self.c_proj(out))
        return out

# Train MQA
mqa_results = train_model("Multi-Query Attention (MQA)", MultiQueryAttention)
results_registry.append(mqa_results)


--- Training Model with Multi-Query Attention (MQA) ---
Config: block_size=256, batch_size=64, d_model=384, layers=6
step 0: train loss 5.1362, val loss 5.1312
step 250: train loss 2.4257, val loss 2.4413
step 500: train loss 2.2666, val loss 2.2875
step 750: train loss 1.8545, val loss 1.9133
step 1000: train loss 1.5809, val loss 1.6689
step 1250: train loss 1.4287, val loss 1.5631
step 1499: train loss 1.3271, val loss 1.4807
Training completed in 736.82 seconds.
Final Val Loss: 1.4959 | Perplexity: 4.4632

Sample Output:
   Fle Spem as an exculects , and to thus templement to remained dulater thats average wils were rived


In [9]:
# # Cell 9: Grouped-Query Attention (GQA)

# class GroupedQueryAttention(nn.Module):
#     """ Grouped-Query Attention (GQA) - Intermediate between MHA and MQA.
#         We group queries into chunks, and each chunk shares a single KV head.
#     """

#     def __init__(self, n_heads, n_embd, n_kv_heads=2):
#         # n_kv_heads must divide n_heads. Using 2 KV heads for our 4 Query heads.
#         super().__init__()
#         assert n_embd % n_heads == 0, "Embedding dimension must be divisible by number of heads"
#         assert n_heads % n_kv_heads == 0, "Number of heads must be divisible by number of KV heads"

#         self.n_heads = n_heads
#         self.n_kv_heads = n_kv_heads
#         self.d_k = n_embd // n_heads # Head dimension
#         self.n_rep = n_heads // n_kv_heads # How many query heads share a single KV head

#         # Query projection (all heads)
#         self.c_q = nn.Linear(n_embd, n_heads * self.d_k, bias=False)
#         # Key/Value projection (fewer heads)
#         self.c_kv = nn.Linear(n_embd, 2 * n_kv_heads * self.d_k, bias=False)
#         self.c_proj = nn.Linear(n_embd, n_embd)

#         self.attn_dropout = nn.Dropout(dropout)
#         self.resid_dropout = nn.Dropout(dropout)

#         # Create a causal lower-triangular mask
#         self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
#                                     .view(1, 1, block_size, block_size))

#     def forward(self, x):
#         B, T, C = x.size()

#         # Calculate Queries: (B, T, C) -> (B, T, n_heads, d_k)
#         q = self.c_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

#         # Calculate Keys & Values: (B, T, 2 * n_kv_heads * d_k)
#         kv = self.c_kv(x)
#         k, v = kv.split(self.n_kv_heads * self.d_k, dim=2)

#         # Reshape to (B, n_kv_heads, T, d_k)
#         k = k.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
#         v = v.view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)

#         # GQA crucial step: Repeat Keys and Values to match the number of Query heads
#         # k: (B, n_kv_heads, T, d_k) -> (B, n_kv_heads, 1, T, d_k) -> expand -> reshape to (B, n_heads, T, d_k)
#         k = k.unsqueeze(2).expand(B, self.n_kv_heads, self.n_rep, T, self.d_k).reshape(B, self.n_heads, T, self.d_k)
#         v = v.unsqueeze(2).expand(B, self.n_kv_heads, self.n_rep, T, self.d_k).reshape(B, self.n_heads, T, self.d_k)

#         # Standard attention map calculation (now perfectly dimension matched)
#         wei = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_k))
#         wei = wei.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
#         wei = F.softmax(wei, dim=-1)
#         wei = self.attn_dropout(wei)

#         out = wei @ v
#         out = out.transpose(1, 2).contiguous().view(B, T, C)
#         out = self.resid_dropout(self.c_proj(out))
#         return out

# # Train GQA
# gqa_results = train_model("Grouped-Query Attention (GQA)", GroupedQueryAttention)
# results_registry.append(gqa_results)

In [10]:
# # Cell 10: Sliding Window (Local) Attention

# class SlidingWindowAttention(nn.Module):
#     """ Sliding Window (Local) Attention - Restricts the attention span to a fixed window size W """

#     def __init__(self, n_heads, n_embd, window_size=64): # Increased window_size for T4
#         super().__init__()
#         assert n_embd % n_heads == 0, "Embedding dimension must be divisible by number of heads"
#         self.n_heads = n_heads
#         self.d_k = n_embd // n_heads
#         self.window_size = window_size

#         self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
#         self.c_proj = nn.Linear(n_embd, n_embd)

#         self.attn_dropout = nn.Dropout(dropout)
#         self.resid_dropout = nn.Dropout(dropout)

#         # Create a banded causal mask
#         mask = torch.tril(torch.ones(block_size, block_size))
#         # Keep only the 'window_size' most recent tokens (including the current token)
#         mask = torch.triu(mask, diagonal=-window_size + 1)
#         self.register_buffer("bias", mask.view(1, 1, block_size, block_size))

#     def forward(self, x):
#         B, T, C = x.size()

#         qkv = self.c_attn(x)
#         q, k, v = qkv.split(d_model, dim=2)

#         k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
#         q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
#         v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

#         wei = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_k))

#         # Apply the banded causal mask
#         wei = wei.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
#         wei = F.softmax(wei, dim=-1)
#         wei = self.attn_dropout(wei)

#         out = wei @ v
#         out = out.transpose(1, 2).contiguous().view(B, T, C)
#         out = self.resid_dropout(self.c_proj(out))
#         return out

# # Train Sliding Window Attention
# swa_results = train_model("Sliding Window Attention (W=64)", SlidingWindowAttention)
# results_registry.append(swa_results)

In [13]:
# Cell 11: Linear (Kernelized) Attention

class LinearAttention(nn.Module):
    """ Linear Attention - Replaces Softmax with an ELU+1 kernel mapping for O(N) complexity """

    def __init__(self, n_heads, n_embd):
        super().__init__()
        assert n_embd % n_heads == 0, "Embedding dimension must be divisible by number of heads"
        self.n_heads = n_heads
        self.d_k = n_embd // n_heads

        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd)

        self.resid_dropout = nn.Dropout(dropout)

        # We do not need a boolean causal mask here! We use cumulative sums instead.

    def feature_map(self, x):
        # Softmax requires exponentiation. Linear attention uses a positive kernel mapping.
        # ELU(x) + 1 is a common choice that keeps all values positive without explosions.
        return F.elu(x) + 1.0

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.c_attn(x)
        q, k, v = qkv.split(d_model, dim=2)

        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Apply the feature map to ensure all keys and queries are positive
        q = self.feature_map(q)
        k = self.feature_map(k)

        # --- Linear Casual Attention Logic (Cumulative sums) ---
        # Instead of multiplying (Q @ K^T) @ V (which is O(N^2)),
        # we calculate K^T @ V first, then Q @ (K^T V) (which is O(N)).

        # For causal masking in Linear Attention, we compute cumulative sums over Time.
        # k_v_prod: (B, n_heads, T, d_k, d_k) - essentially a running "memory state"
        k_v_prod = torch.einsum('b h t d, b h t c -> b h t d c', k, v)
        kv_cumsum = torch.cumsum(k_v_prod, dim=2)

        # The denominator ensures probabilities sum to 1 (acting like the Softmax partition function)
        k_cumsum = torch.cumsum(k, dim=2)
        denominator = torch.einsum('b h t d, b h t d -> b h t', q, k_cumsum) + 1e-6

        # The numerator is the current Query multiplied by the historical KV state
        numerator = torch.einsum('b h t d, b h t d c -> b h t c', q, kv_cumsum)

        # Final output
        out = numerator / denominator.unsqueeze(-1)

        # Re-assemble
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.c_proj(out))
        return out


# --- HYPERPARAMETER OVERRIDE FOR LINEAR ATTENTION ---
# Linear Attention uses O(N) complexity instead of O(N^2). To realistically show
# its architectural benefits over standard MHA, we dynamically increase its window size by 4x!
old_batch_size = batch_size
old_block_size = block_size

batch_size = 16    # We scale down batch_size so it still fits inside T4 VRAM
block_size = 512  # We scale up block_size by 4x to show off Linear Attention's memory capabilities

linear_results = train_model("Linear Attention (512 Context Window)", LinearAttention)
results_registry.append(linear_results)

# Restore standard hyperparameters for safety
batch_size = old_batch_size
block_size = old_block_size


--- Training Model with Linear Attention (1024 Context Window) ---
Config: block_size=512, batch_size=16, d_model=384, layers=6
step 0: train loss 5.0404, val loss 5.0475
step 250: train loss 2.6507, val loss 2.6323
step 500: train loss 2.4368, val loss 2.4424
step 750: train loss 2.3760, val loss 2.3859
step 1000: train loss 2.3173, val loss 2.3370
step 1250: train loss 2.2265, val loss 2.2693
step 1499: train loss 2.0257, val loss 2.0846
Training completed in 1115.63 seconds.
Final Val Loss: 2.0745 | Perplexity: 7.9610

Sample Output:
   Chirdilecedied the rening ponionga blacows recciagitenal ox , witition  th and ain estauly the a 10


In [14]:
# Cell 12: Final Evaluation and Metric Comparison

import pandas as pd
from IPython.display import display

def display_results(results):
    # Convert list of dictionaries to DataFrame
    df = pd.DataFrame(results)

    # Add a blank column for your human evaluation
    df["Human Eval (1-5)"] = ""

    # Format the numerical columns for readability
    df["Val Loss"] = df["Val Loss"].map("{:.4f}".format)
    df["Perplexity"] = df["Perplexity"].map("{:.2f}".format)
    df["Time (s)"] = df["Time (s)"].map("{:.2f}".format)

    # Arrange columns safely to show the specific Hyperparameters used
    cols = ["Model", "Block Size", "Batch Size", "Val Loss", "Perplexity", "Time (s)", "Human Eval (1-5)"]
    df = df[[c for c in cols if c in df.columns]]

    # Sort by execution time (fastest first) as an example
    df = df.sort_values(by="Time (s)")
    df = df.reset_index(drop=True)

    print("\n================ FINAL EXPERIMENT RESULTS (T4 V2) ================\n")
    display(df)

    print("\nObservation Guide for your Report:")
    print("- We scaled hyperparameters up drastically for the Colab T4!")
    print("- MHA should achieve excellent loss, but takes significant memory.")
    print("- MQA/GQA should be structurally identical to MHA but use drastically less KV cache (great for inference).")
    print("- Linear Attention was dynamically given a massively extended block_size (512 context window)")
    print("  to demonstrate its ability to train on huge contexts without OOMing the GPU.")

display_results(results_registry)


================ FINAL EXPERIMENT RESULTS (T4 V2) ================



,Model,Block Size,Batch Size,Val Loss,Perplexity,Time (s),Human Eval (1-5)
0,Linear Attention (1024 Context Window),512,16,2.0745,7.96,1115.63,
1,Multi-Query Attention (MQA),256,64,1.4959,4.46,736.82,
2,Multi-Head Attention (Baseline),256,64,1.4637,4.32,815.03,



Observation Guide for your Report:
- We scaled hyperparameters up drastically for the Colab T4!
- MHA should achieve excellent loss, but takes significant memory.
- MQA/GQA should be structurally identical to MHA but use drastically less KV cache (great for inference).
- Linear Attention was dynamically given a massively extended block_size (1024 context window)
  to demonstrate its ability to train on huge contexts without OOMing the GPU.
